# Lesson 25: Transform Composition Patterns

            **Phase:** Integration Labs  
            **Script:** `lessons/lesson_25_transform_composition.py`  
            **Exercise:** `exercise_clip_gradients`  
            **Reference solution:** `solutions.lesson_25_solution.clip_gradients`


## What This Lesson Teaches
- Compose `vmap`, `grad`, and `jit` without changing the mathematical contract.
- Compare per-example gradients with gradients of a reduced batch objective.
- Use a compiled update as an integration point for earlier transform lessons.

## Mental Model
Transform composition works best when each function has a crisp contract: one example, one batch, one loss, or one update.


In [1]:
from pathlib import Path

import jax
import jax.numpy as jnp

import lessons.lesson_25_transform_composition as lesson
import solutions.lesson_25_solution as solution


## Guided Demo
Run the cell below before attempting the exercise. It prints compact evidence for the key shapes, values, or state transitions.


In [2]:
x, y = lesson.make_regression_batch()
params = lesson.init_params()
grads = lesson.per_example_gradients(params, x, y)
updated, loss = lesson.compiled_update(params, x, y, lr=0.1)
clipped = solution.clip_gradients({'w': jnp.array([3.0, 4.0]), 'b': jnp.array(0.0)}, max_norm=2.0)
print('loss=', round(float(loss), 4), 'updated_keys=', sorted(updated.keys()))
print('per_example_grad_shapes=', {k: v.shape for k, v in grads.items()})
print('clipped_norm=', round(float(jnp.sqrt(sum(jnp.sum(v ** 2) for v in clipped.values()))), 4))


loss= 1.6725 updated_keys= ['b', 'w']
per_example_grad_shapes= {'b': (4,), 'w': (4, 2)}
clipped_norm= 2.0


## Workbook Exercise
TODO: `exercise_clip_gradients`. Clip a gradient pytree by global L2 norm while preserving its structure.

Hint: Compute the global norm from all leaves, form a scale no larger than one, then tree-map the scale over the leaves.

The next cell prints the exercise name, reference solution path, and ready status so this published notebook stays executable. Replace it with your own scratch implementation while studying.


In [3]:
print('exercise:', 'exercise_clip_gradients')
print('reference:', 'solutions.lesson_25_solution.clip_gradients')
print('status: ready for student implementation')


exercise: exercise_clip_gradients
reference: solutions.lesson_25_solution.clip_gradients
status: ready for student implementation


## Expected Output Checkpoint
- Per-example gradient leaves keep the batch dimension first.
- The compiled update returns the same parameter tree keys as the input.
- Clipped gradients have global norm no larger than the requested limit.


## Common Mistakes
- Changing the reduction axis and accidentally changing what the gradient means.
- Compiling a large opaque training step before checking the small eager pieces.

## Extension
Swap the order of `jit`, `vmap`, and `grad` for a tiny function and inspect the resulting JAXPR.
